# Validation v4 — er_pipeline v3.0 (CPU only, no embeddings)

**What changed vs v3.1** (details in the `er_pipeline.py` docstring):
- **A. Realistic training rows.** S2/S3 records whose true S1 is outside the 15% selection ("orphans") are dropped from training and scoring. Early stopping, fold models saved.
- **B. Candidate expansion.** After stage 1, S2/S3 records sharing a non-generic exact name / glued-name / full-address key with an S1 or its confident matches become new candidates.
- **C. 22 new stage-1 features + 3 new stage-2 features.** Synthetic-name detector, glued-name split, exact address, private/public legal conflicts, name-duplicate counts, expansion flags.
- **D. Decision.** Expected-F0.5 per S1 vs the old global threshold; the better one is saved for test.

**Reuses** `prep/`, `fillers_train.json`, `selected.parquet`, `blocks/`, `features/`, `features_extra/` from your v3.0/v3.1 run. New work: `features_v3/` (CPU), stage-1/2 training, expansion.

Run all cells in order. Every step is cached under `RUN_DIR/v4/`; after a disconnect re-run cells 1–3 and continue.

In [1]:
# ===== CELL 1: install =====
!pip -q install lightgbm rapidfuzz sparse_dot_topn indic_transliteration
import sys, os, json, gc, importlib, time, glob, shutil
import numpy as np, pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.6/309.6 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 15.6 MB/s eta 0:00:00


In [2]:
# ===== CELL 2: configuration (Google Colab + Drive) =====
from google.colab import drive
drive.mount("/content/drive")
INPUT_BASE = "/content/drive/MyDrive/amazon ml new arch"   # train/, test/, new_parquet/, er_work/, er_pipeline.py
BASE       = INPUT_BASE
CODE_DIR   = INPUT_BASE
PARQUET_DIR = f"{INPUT_BASE}/new_parquet"
INDIC_DICT  = f"{PARQUET_DIR}/indic_dict.json"
WORK_DIR    = f"{INPUT_BASE}/er_work"                   # your cache; new files are written here too

FRAC, SEED   = 0.15, 42      # same as before -> blocks/features caches are reused
N_FOLDS      = 3
MAX_ROUNDS   = 1500          # early stopping picks the real number (was fixed 400)
EARLY_STOP   = 100
MIN_P        = 0.02          # stage-2 rows: p1 >= this (was 0.05)
CONF_EXP     = 0.7           # confident-match threshold that seeds candidate expansion
FEAT_WORKERS = max(1, min(12, os.cpu_count() // 2))

sys.path.insert(0, CODE_DIR)
import er_pipeline as er
importlib.reload(er)
assert er.VERSION >= "3.0", f"expected er_pipeline.py v3.0, got {er.VERSION} (CODE_DIR={CODE_DIR})"
RUN_DIR = f"{WORK_DIR}/run_frac{FRAC}_seed{SEED}"
V4_DIR  = f"{RUN_DIR}/v4"; os.makedirs(V4_DIR, exist_ok=True)
FINAL_DIR = f"{WORK_DIR}/final_v4"; os.makedirs(FINAL_DIR, exist_ok=True)
assert os.path.exists(INDIC_DICT), f"missing {INDIC_DICT}"
for k, p in {"prep": f"{WORK_DIR}/prep/train_S3.parquet", "selection": f"{RUN_DIR}/selected.parquet",
             "blocks": f"{RUN_DIR}/blocks", "52 base features": f"{RUN_DIR}/features",
             "17 extra features": f"{RUN_DIR}/features_extra", "22 v3 features": f"{RUN_DIR}/features_v3"}.items():
    print(("REUSE   " if os.path.exists(p) else "COMPUTE ") + f"{k:18s} {p}")
print("module", er.VERSION, "| cores", os.cpu_count(), "| feature workers", FEAT_WORKERS)
!free -h

Mounted at /content/drive
REUSE   prep               /content/drive/MyDrive/amazon ml new arch/er_work/prep/train_S3.parquet
REUSE   selection          /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/selected.parquet
REUSE   blocks             /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/blocks
REUSE   52 base features   /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/features
REUSE   17 extra features  /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/features_extra
REUSE   22 v3 features     /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/features_v3
module 3.0 | cores 24 | feature workers 12
               total        used        free      shared  buff/cache   available
Mem:            47Gi       2.1Gi        33Gi       4.7Mi        12Gi        44Gi
Swap:             0B          0B          0B


In [3]:
# ===== CELL 3: load train selection, candidates, labels + v3 resources =====
t0 = time.time()
er.prepare_split("train", INPUT_BASE, WORK_DIR, INDIC_DICT, parquet_dir=PARQUET_DIR)
fp = f"{WORK_DIR}/fillers_train.json"
if not os.path.exists(fp):
    json.dump(er.detect_fillers("train", WORK_DIR), open(fp, "w"))
fillers = json.load(open(fp))
sp = f"{RUN_DIR}/selected.parquet"
if os.path.exists(sp):
    S = pd.read_parquet(sp); parts = json.load(open(f"{RUN_DIR}/partitions.json"))
    for c in ("src", "country"): S[c] = S[c].astype("category")
else:
    S, parts = er.load_selection("train", WORK_DIR, frac=FRAC, seed=SEED, fillers=fillers)
    S.to_parquet(sp, index=False); json.dump(parts, open(f"{RUN_DIR}/partitions.json", "w"))
S = er.attach_full_tokens(S, "train", WORK_DIR)
S, SYLL = er.setup_v3_resources(S, "train", INPUT_BASE, WORK_DIR, PARQUET_DIR)   # lseq, name counts, syllables, split vocab
C = er.run_blocking(S, parts, RUN_DIR, n_workers=3)
res = er.build_resources(S, fillers)
gt = er.load_ground_truth(INPUT_BASE, PARQUET_DIR)
y = er.label_pairs(C, S, gt)
print(f"S1 {int((S.src == 'S1').sum()):,} | candidates {len(C):,} | positives {int(y.sum()):,} | {(time.time() - t0) / 60:.1f} min")
print("syllable inventory:", len(SYLL), SYLL[:60])
syn = [t for t in pd.Series(" ".join(S.ftoks.values[(S.src != "S1").values][:200000]).split()).unique() if er.is_synth(t)]
print("sample tokens flagged synthetic:", syn[:40])

[13:03:39] [prepare train] Indic dictionary entries: 1,302 | workers: 24
[13:03:39] [prepare train] S1 cached
[13:03:42] [prepare train] S2 cached
[13:03:42] [prepare train] S3 cached
[13:03:42] [prepare train] done
[13:04:14] [v3 resources train] syllables=123 | split vocab=64,024 | 24s
[13:04:15] [block] 9 jobs cached, 0 to run | workers=3 | kNN threads/job=8
S1 336,920 | candidates 11,514,059 | positives 1,149,321 | 1.1 min
syllable inventory: 123 ['arc', 'arcio', 'arcx', 'ariaio', 'ariax', 'avi', 'aviio', 'avix', 'belo', 'beloio', 'belox', 'brix', 'brixio', 'brixx', 'calo', 'caloio', 'calox', 'cira', 'ciraio', 'cirax', 'delta', 'deltax', 'dova', 'dovaio', 'dovax', 'drex', 'drexio', 'drexx', 'ecto', 'ectoio', 'ectox', 'evo', 'evoio', 'evox', 'faye', 'fayeio', 'fayex', 'flux', 'fluxio', 'fluxx', 'gild', 'gildio', 'gildx', 'halo', 'haloio', 'halox', 'iri', 'iriio', 'irix', 'jax', 'jaxio', 'jaxx', 'kelo', 'keloio', 'kelox', 'kor', 'korio', 'korx', 'lum', 'lumio']
sample tokens flagged 

In [ ]:
# ===== EXPORT CELL (after cell 3) ===== 2nd newly added ==========
HOLD = 2
mask = er.realistic_mask(C, S, gt)
CE = er.concat_candidates(C[mask].reset_index(drop=True), pd.read_parquet(f"{V4_DIR}/E.parquet"))
p = np.load(f"{V4_DIR}/p_final.npy"); assert len(p) == len(CE)
y = er.label_pairs(CE, S, gt)
fold = er.s1_folds(S, N_FOLDS, SEED)[CE.i1.values]
eid = S.entity_id.values; cty = S.country.astype(str).values[CE.i1.values]
rng = np.random.RandomState(0)
band = (p >= 0.005) & (p <= 0.995)
sel = np.flatnonzero(band | ((p < 0.005) & (rng.rand(len(p)) < 0.01)) | ((p > 0.995) & (rng.rand(len(p)) < 0.10))
                     | ((fold == HOLD) & (p >= 0.005)))
pd.DataFrame({"row": sel.astype(np.int64), "s1_id": eid[CE.i1.values[sel]], "pool_id": eid[CE.i2.values[sel]],
              "p": p[sel], "y": y[sel].astype(np.int8), "fold": fold[sel].astype(np.int8),
              "country": cty[sel], "band": band[sel]}).to_parquet(f"{V4_DIR}/ce_val_pairs.parquet", index=False)
print(f"exported {len(sel):,} pairs -> {V4_DIR}/ce_val_pairs.parquet")
print("train rows:", int((fold[sel] != HOLD).sum()), "| hold rows:", int((fold[sel] == HOLD).sum()))

[07:37:00] [realistic] orphan S2/S3 records in selection: 362,698 | candidate rows dropped: 1,319,526 of 11,514,059
exported 761,926 pairs -> /content/drive/MyDrive/amazon ml new arch/er_work/run_frac0.15_seed42/v4/ce_val_pairs.parquet
train rows: 295486 | hold rows: 466440


In [4]:
####################################################################################################
# [2] er_validation_v4.ipynb -> add after cell 3 (replaces the earlier [B]); run cells 1-3 + this, AFTER the Kaggle large scores are on Drive
####################################################################################################
# ===== STACK CELL: er_validation_v4, after cell 3 -> picks best of LightGBM / blends / stacker over ALL cross-encoders =====


#===================3rd newly added ====================


def _logit(x):
    x = np.clip(np.asarray(x, np.float64), 1e-6, 1 - 1e-6)
    return np.log(x / (1 - x))

# cross-encoders used when BOTH their files are in er_work/ce/ (base from Colab, large from Kaggle)
CE_SOURCES = [("base", "ce_val_scores.parquet", "ce_test_scores.parquet"),
              ("large", "ce_large_val_scores.parquet", "ce_large_test_scores.parquet")]
STACK_COLS = None

def stack_features(i1, i2, p_row, ce_list, all_i2, all_p, src_b, addr_b):
    """Stacker inputs for the scored pairs. ce_list: one probability array per cross-encoder (same rows)."""
    global STACK_COLS
    lp = _logit(p_row); lcs = [_logit(c) for c in ce_list]; lcm = np.mean(lcs, axis=0)
    cols = {"lp": lp, "lcm": lcm, "lb": 0.5 * (lp + lcm)}
    for k, lc in enumerate(lcs):
        cols[f"lc{k}"] = lc
    d = pd.DataFrame({"i1": i1, **cols})
    F = dict(cols); F["diff"] = lcm - lp
    g = d.groupby("i1", sort=False)
    F["n_cand"] = g.lp.transform("size").values
    for c in cols:
        r = g[c].rank(ascending=False, method="first").values
        mx = g[c].transform("max").values
        F[f"{c}_rank"] = r; F[f"{c}_gap"] = mx - d[c].values; F[f"{c}_max"] = mx
        F[f"{c}_npos"] = (d[c] > 0).groupby(d.i1, sort=False).transform("sum").values
        F[f"{c}_second"] = d[c].where(r == 2).groupby(d.i1, sort=False).transform("max").fillna(-14).values
    keep = np.isin(all_i2, np.unique(i2)) & (all_p >= 0.001)
    a = pd.DataFrame({"i2": all_i2[keep], "p": all_p[keep]}).sort_values(["i2", "p"], ascending=[True, False])
    rk = a.groupby("i2", sort=False).cumcount().values
    best = a[rk == 0].set_index("i2").p; second = a[rk == 1].set_index("i2").p
    nn = a.groupby("i2", sort=False).size(); s_i2 = pd.Series(i2)
    b1 = s_i2.map(best).fillna(0).values; b2 = s_i2.map(second).fillna(0).values
    other = np.where(p_row >= b1, b2, b1)
    F["pool_best_other"] = other; F["pool_margin"] = p_row - other
    F["pool_is_top"] = (p_row >= b1).astype(np.float32); F["pool_n"] = s_i2.map(nn).fillna(1).values
    F["b_s3"] = src_b.astype(np.float32); F["b_has_addr"] = addr_b.astype(np.float32)
    STACK_COLS = list(F)
    return np.column_stack([np.asarray(F[k], np.float32) for k in STACK_COLS])

STACK_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100,
                    feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=42)

import lightgbm as lgb
HOLD = 2; CE_DIR = f"{WORK_DIR}/ce"
mask = er.realistic_mask(C, S, gt)
CE = er.concat_candidates(C[mask].reset_index(drop=True), pd.read_parquet(f"{V4_DIR}/E.parquet"))
p = np.load(f"{V4_DIR}/p_final.npy"); assert len(p) == len(CE)
FOLD_S1 = er.s1_folds(S, N_FOLDS, SEED)
posH = np.flatnonzero(FOLD_S1[CE.i1.values] == HOLD)
CEh = CE.iloc[posH].reset_index(drop=True); ph = p[posH]
srcs = [(n, v) for n, v, t in CE_SOURCES if os.path.exists(f"{CE_DIR}/{v}") and os.path.exists(f"{CE_DIR}/{t}")]
assert srcs, f"no cross-encoder score files in {CE_DIR}"
ce_cols = []
for n, v in srcs:
    cv = pd.read_parquet(f"{CE_DIR}/{v}")
    loc = np.searchsorted(posH, cv.row.values)
    assert (posH[np.clip(loc, 0, len(posH) - 1)] == cv.row.values).all(), f"{v}: rows do not match - wrong files?"
    col = np.full(len(ph), np.nan); col[loc] = cv.ce.values; ce_cols.append(col)
has = np.flatnonzero(np.all([~np.isnan(c) for c in ce_cols], axis=0))
ce_has = [c[has] for c in ce_cols]
G = er.gt_arrays(S, gt); kt = FOLD_S1[G["ti1"]] == HOLD
Gh = dict(G, s1_rows=G["s1_rows"][FOLD_S1[G["s1_rows"]] == HOLD], ti1=G["ti1"][kt], ti2=G["ti2"][kt])
yh = er.label_pairs(CEh, S, gt)
i1h, i2h = CEh.i1.values, CEh.i2.values
src_s3 = (S.src == "S3").values; addr_ok = S.has_addr.values.astype(bool)
print(f"cross-encoders used: {[n for n, _ in srcs]} | hold fold: {len(Gh['s1_rows']):,} S1 | "
      f"{len(CEh):,} rows | {len(has):,} with scores")

Xs = stack_features(i1h[has], i2h[has], ph[has], ce_has, i2h, ph, src_s3[i2h[has]], addr_ok[i2h[has]])
ys = yh[has]; inner = (i1h[has] * 2654435761 % 2**32) % 4
oof = np.zeros(len(has), np.float32)
for f in range(4):
    tr_, va_ = inner != f, inner == f
    m = lgb.train(STACK_PARAMS, lgb.Dataset(Xs[tr_], ys[tr_], feature_name=STACK_COLS), num_boost_round=400)
    oof[va_] = m.predict(Xs[va_])
print("stacker out-of-fold predictions done")

decs = [{"method": "expected", "gamma": 1.0, "lam": l, "miss": mm} for l in (-0.05, 0.0, 0.05, 0.1, 0.2) for mm in (0.0, 0.25)]
lcm = np.mean([_logit(c) for c in ce_has], axis=0)
cands = {}
for w in (1.0, 0.75, 0.6, 0.5, 0.4, 0.25):
    lg = _logit(ph); lg[has] = w * lg[has] + (1 - w) * lcm
    cands[f"blend w={w}"] = (1 / (1 + np.exp(-lg))).astype(np.float32)
pst = ph.copy(); pst[has] = oof; cands["stacker"] = pst
rows = []
for name, pp in cands.items():
    for d in decs:
        rows.append({"model": name, **d, "F05": er.score_keep(CEh, er.apply_decision(CEh, pp, d, S), Gh)["F05"]})
grid = pd.DataFrame(rows).sort_values("F05", ascending=False).reset_index(drop=True)
best_by = grid.groupby("model").F05.max().sort_values(ascending=False)
print(best_by.round(6).to_string())
b = grid.iloc[0]; base = best_by["blend w=1.0"]
dec = {"method": "expected", "gamma": 1.0, "lam": float(b.lam), "miss": float(b.miss)}
bp = {"model": b.model, "w": float(b.model.split("=")[1]) if b.model.startswith("blend") else None, "decision": dec,
      "ce_used": [n for n, _ in srcs], "F05_hold_best": float(b.F05), "F05_hold_lgbm_only": float(base)}
if b.model == "stacker":
    m = lgb.train(STACK_PARAMS, lgb.Dataset(Xs, ys, feature_name=STACK_COLS), num_boost_round=400)
    m.save_model(f"{CE_DIR}/stacker.txt")
json.dump(bp, open(f"{CE_DIR}/blend_params.json", "w"), indent=1)
print(f"\nBEST: {b.model} | HOLD F0.5 LightGBM only {base:.6f} -> {b.F05:.6f} ({b.F05 - base:+.6f})")
print("saved", f"{CE_DIR}/blend_params.json", bp)

[13:05:14] [realistic] orphan S2/S3 records in selection: 362,698 | candidate rows dropped: 1,319,526 of 11,514,059
cross-encoders used: ['base', 'large'] | hold fold: 111,967 S1 | 3,501,280 rows | 466,440 with scores
stacker out-of-fold predictions done
model
stacker         0.991142
blend w=0.6     0.991135
blend w=0.5     0.991063
blend w=0.4     0.990678
blend w=0.75    0.990462
blend w=0.25    0.989783
blend w=1.0     0.987119

BEST: stacker | HOLD F0.5 LightGBM only 0.987119 -> 0.991142 (+0.004023)
saved /content/drive/MyDrive/amazon ml new arch/er_work/ce/blend_params.json {'model': 'stacker', 'w': None, 'decision': {'method': 'expected', 'gamma': 1.0, 'lam': 0.1, 'miss': 0.0}, 'ce_used': ['base', 'large'], 'F05_hold_best': 0.9911416088841192, 'F05_hold_lgbm_only': 0.9871185902306773}


In [ ]:
# ===== CELL 4: A - realistic rows (drop orphans) + stage-1 features =====
mask = er.realistic_mask(C, S, gt)
print(f"positives kept: {int(y[mask].sum()):,} of {int(y.sum()):,} (should be equal - orphans are never true pairs here)")
t0 = time.time()
Xv = er.build_features(C, S, res, RUN_DIR, mode="v3", n_workers=FEAT_WORKERS)     # new, ~CPU only
print(f"v3 block: {Xv.shape} | {(time.time() - t0) / 60:.1f} min")
X1 = np.hstack([er.build_features(C, S, res, RUN_DIR, mode="base").values[mask],
                er.build_features(C, S, res, RUN_DIR, mode="extra").values[mask],
                Xv.values[mask]]).astype(np.float32)
del Xv; gc.collect()
Cm = C[mask].reset_index(drop=True); y1 = y[mask]
FOLD_S1 = er.s1_folds(S, N_FOLDS, SEED)
assert X1.shape[1] == len(er.FEATURES_S1)
print("stage-1 matrix:", X1.shape)
!free -h

[19:22:55] [realistic] orphan S2/S3 records in selection: 362,698 | candidate rows dropped: 1,319,526 of 11,514,059
positives kept: 1,149,321 of 1,149,321 (should be equal - orphans are never true pairs here)
[19:23:23] [features:v3] part 1/24 (18s)
[19:23:24] [features:v3] part 2/24 (19s)
[19:23:25] [features:v3] part 3/24 (20s)
[19:23:26] [features:v3] part 4/24 (21s)
[19:23:27] [features:v3] part 5/24 (22s)
[19:23:28] [features:v3] part 6/24 (23s)
[19:23:29] [features:v3] part 7/24 (24s)
[19:23:30] [features:v3] part 8/24 (26s)
[19:23:31] [features:v3] part 9/24 (27s)
[19:23:33] [features:v3] part 10/24 (28s)
[19:23:34] [features:v3] part 11/24 (29s)
[19:23:35] [features:v3] part 12/24 (30s)
[19:23:35] [features:v3] part 13/24 (30s)
[19:23:40] [features:v3] part 14/24 (35s)
[19:23:40] [features:v3] part 15/24 (35s)
[19:23:41] [features:v3] part 16/24 (36s)
[19:23:41] [features:v3] part 17/24 (36s)
[19:23:41] [features:v3] part 18/24 (37s)
[19:23:42] [features:v3] part 19/24 (37s)
[1

In [ ]:
# ===== CELL 5: stage 1 - grouped CV with early stopping =====
op = f"{V4_DIR}/oof_s1.npy"
cv1 = er.cv_train(X1, y1, FOLD_S1[Cm.i1.values], Cm.i1.values, er.FEATURES_S1, max_rounds=MAX_ROUNDS,
                  early_stop=EARLY_STOP, model_dir=V4_DIR, tag="s1")
p1 = cv1["oof"]; np.save(op, p1)
json.dump({"s1_iters": cv1["iters"]}, open(f"{V4_DIR}/iters_s1.json", "w"))
print(cv1["imp"].head(30).round(0).to_string())
print("\nv3 feature ranks:", {f: int((cv1["imp"].index == f).argmax()) + 1 for f in er.FEATURES_V3})

[250]	valid_0's binary_logloss: 0.00807005
[500]	valid_0's binary_logloss: 0.00756593
[750]	valid_0's binary_logloss: 0.00745716
[1000]	valid_0's binary_logloss: 0.00744866
[19:42:55] [cv s1] fold 1/3: best_iter 942, predict 3,404,654 (1053s)
[250]	valid_0's binary_logloss: 0.00817362
[500]	valid_0's binary_logloss: 0.00771331
[750]	valid_0's binary_logloss: 0.00760026
[19:59:40] [cv s1] fold 2/3: best_iter 897, predict 3,401,048 (1005s)
[250]	valid_0's binary_logloss: 0.00819328
[500]	valid_0's binary_logloss: 0.00767572
[750]	valid_0's binary_logloss: 0.00757294
[1000]	valid_0's binary_logloss: 0.00755051
[20:17:40] [cv s1] fold 3/3: best_iter 979, predict 3,388,831 (1080s)
r2                   58895698.0
gap2                 23324679.0
num_b_unmatched       5857511.0
extra_b_full          2142322.0
num_first_absdiff     1383866.0
num_b_subset_a        1277093.0
num_set_jacc          1163806.0
b_oov_frac             934773.0
r1                     681630.0
lseq_diff              6252

In [ ]:
# ===== CELL 6: B - candidate expansion (+ its stage-1 features and fold predictions) =====
ep = f"{V4_DIR}/E.parquet"
if os.path.exists(ep):
    E = pd.read_parquet(ep); XE = np.load(f"{V4_DIR}/XE.npy")
else:
    E = er.expand_candidates(Cm, S, p1, conf=CONF_EXP)
    E = E[er.realistic_mask(E, S, gt)].reset_index(drop=True)
    XE = er.features_for_new(C, E, S, res, n_workers=FEAT_WORKERS)   # competition over C_full + E, as on test
    E.to_parquet(ep, index=False); np.save(f"{V4_DIR}/XE.npy", XE)
yE = er.label_pairs(E, S, gt)
fE = FOLD_S1[E.i1.values]
p1E = np.zeros(len(E), np.float32)
for f, m in enumerate(cv1["models"]):
    if (fE == f).any():
        p1E[fE == f] = m.predict(XE[fE == f], num_iteration=cv1["iters"][f])
print(f"expansion rows {len(E):,} | true pairs among them {int(yE.sum()):,} | "
      f"true pairs with p1 >= 0.5: {int((yE[p1E >= 0.5]).sum()):,} | false with p1 >= 0.5: {int(((1 - yE)[p1E >= 0.5]).sum()):,}")

[20:18:15] [expand] seeds: 1,132,283 confident pairs + 336,920 S1 | new pairs 431,718 (via sibling only 228,972, address key 76,290) (35s)
[20:18:18] [realistic] orphan S2/S3 records in selection: 362,698 | candidate rows dropped: 95,041 of 431,718
expansion rows 336,677 | true pairs among them 2,965 | true pairs with p1 >= 0.5: 1,996 | false with p1 >= 0.5: 212


In [ ]:
# ===== CELL 7: stage 2 over C + E (sibling features) =====
CE = er.concat_candidates(Cm, E); nCm = len(Cm)
p1CE = np.r_[p1, p1E].astype(np.float32); yCE = np.r_[y1, yE].astype(np.int8)
sp2 = f"{V4_DIR}/sib.npz"
if os.path.exists(sp2):
    z = np.load(sp2); rows, M = z["rows"], z["M"]
else:
    rows, M = er.sibling_features(CE, S, p1CE, min_p=MIN_P)
    np.savez(sp2, rows=rows, M=M)
assert M.shape[1] == len(er.SIB_FEATURES)
rc = rows < nCm
X2 = np.zeros((len(rows), len(er.FEATURES_S2)), np.float32)
X2[rc, :X1.shape[1]] = X1[rows[rc]]; X2[~rc, :X1.shape[1]] = XE[rows[~rc] - nCm]; X2[:, X1.shape[1]:] = M
print(f"stage-2 rows {len(rows):,} ({100 * len(rows) / len(CE):.1f}% of candidates, "
      f"{100 * yCE[rows].sum() / max(yCE.sum(), 1):.2f}% of true candidate pairs)")
cv2 = er.cv_train(X2, yCE[rows], FOLD_S1[CE.i1.values[rows]], CE.i1.values[rows], er.FEATURES_S2,
                  max_rounds=MAX_ROUNDS, early_stop=EARLY_STOP, model_dir=V4_DIR, tag="s2")
json.dump({"s2_iters": cv2["iters"]}, open(f"{V4_DIR}/iters_s2.json", "w"))
p = p1CE.copy(); p[rows] = cv2["oof"]
np.save(f"{V4_DIR}/p_final.npy", p)
print(cv2["imp"].head(25).round(0).to_string())

[20:19:29] [siblings] 1,254,438 rows with p1 >= 0.02 (34s)
stage-2 rows 1,254,438 (11.9% of candidates, 99.89% of true candidate pairs)
[250]	valid_0's binary_logloss: 0.0432098
[20:21:31] [cv s2] fold 1/3: best_iter 304, predict 418,877 (121s)
[250]	valid_0's binary_logloss: 0.0437832
[20:23:40] [cv s2] fold 2/3: best_iter 339, predict 418,386 (129s)
[250]	valid_0's binary_logloss: 0.0422451
[20:25:36] [cv s2] fold 3/3: best_iter 300, predict 417,175 (116s)
p1                   7117721.0
p1_gap1               945894.0
p1_other_best         285779.0
sib_num_eq            149933.0
sib_n                  86537.0
sib_maxp               55244.0
r1                     50596.0
num_b_unmatched        48543.0
gap1                   40922.0
num_b_subset_a         37937.0
p1_r1                  37907.0
num_first_absdiff      36191.0
idf_max_shared         30888.0
b_name_n23             28972.0
sib_addr_jacc          25920.0
n_c2                   24053.0
sib_tsr_max            23519.0
b_nnum    

In [ ]:
# ===== CELL 8: score - ablations + D (decision tuning) =====
G = er.gt_arrays(S, gt)
rep, grid_t, keep_t, detail = er.evaluate(CE, p, S, gt)                       # stage 2 + expansion, threshold
rep_noE, *_ = er.evaluate(Cm, p[:nCm], S, gt)                                # same model, blocking candidates only
p1_only = p1CE.copy()
rep_s1, *_ = er.evaluate(CE, p1_only, S, gt)
dec, dgrid = er.tune_decision(CE, p, S, gt)
keep = er.apply_decision(CE, p, dec, S)
fin = er.score_keep(CE, keep, G)

old = json.load(open(f"{RUN_DIR}/metrics_v3_1.json")) if os.path.exists(f"{RUN_DIR}/metrics_v3_1.json") else {}
tab = pd.DataFrame({
    "v3.1 (old, orphans in)": {"F05": old.get("F05"), "micro_precision": old.get("micro_precision"),
                               "micro_recall": old.get("micro_recall"), "singleton_accuracy": old.get("singleton_accuracy"),
                               "blocking_recall": old.get("blocking_recall"), "ceiling": old.get("F05_perfect_model_on_candidates")},
    "v4 stage 1, tau": {"F05": rep_s1["F05"], "micro_precision": rep_s1["micro_precision"], "micro_recall": rep_s1["micro_recall"],
                        "singleton_accuracy": rep_s1["singleton_accuracy"], "blocking_recall": rep_s1["blocking_recall"],
                        "ceiling": rep_s1["F05_perfect_model_on_candidates"]},
    "v4 stage 2, no expansion, tau": {"F05": rep_noE["F05"], "micro_precision": rep_noE["micro_precision"],
                        "micro_recall": rep_noE["micro_recall"], "singleton_accuracy": rep_noE["singleton_accuracy"],
                        "blocking_recall": rep_noE["blocking_recall"], "ceiling": rep_noE["F05_perfect_model_on_candidates"]},
    "v4 stage 2 + expansion, tau": {"F05": rep["F05"], "micro_precision": rep["micro_precision"], "micro_recall": rep["micro_recall"],
                        "singleton_accuracy": rep["singleton_accuracy"], "blocking_recall": rep["blocking_recall"],
                        "ceiling": rep["F05_perfect_model_on_candidates"]},
    f"v4 FINAL ({dec['method']})": {"F05": fin["F05"], "micro_precision": fin["micro_precision"], "micro_recall": fin["micro_recall"],
                        "singleton_accuracy": fin["singleton_accuracy"], "blocking_recall": rep["blocking_recall"],
                        "ceiling": rep["F05_perfect_model_on_candidates"]}})
print(tab.round(6).to_string())
print("\nfinal F0.5 by country:", {k: round(v, 6) for k, v in fin["F05_by_country"].items()})
print("decision:", dec)
print("\nbest 10 decision settings:"); print(dgrid.head(10).to_string(index=False))

metrics = {**rep, "stage1": rep_s1, "no_expansion": rep_noE, "final": fin, "decision": dec, "min_p": MIN_P,
           "conf_exp": CONF_EXP, "s1_iters": cv1["iters"], "s2_iters": cv2["iters"], "syllables": SYLL,
           "version": er.VERSION}
json.dump(metrics, open(f"{RUN_DIR}/metrics_v4.json", "w"), indent=1)
er.write_id_lists(f"{V4_DIR}/val_matching_results.tsv", "matched_entity_ids", detail[0], CE.i1.values[keep], CE.i2.values[keep], S)
print("\nsaved metrics_v4.json")

                    v3.1 (old, orphans in)  v4 stage 1, tau  v4 stage 2, no expansion, tau  v4 stage 2 + expansion, tau  v4 FINAL (expected)
F05                               0.976169         0.984029                       0.986326                     0.986749             0.987036
micro_precision                   0.991659         0.994318                       0.996660                     0.996576             0.996898
micro_recall                      0.951978         0.966220                       0.968607                     0.970549             0.969549
singleton_accuracy                0.976955         0.977439                       0.989042                     0.988988             0.981414
blocking_recall                   0.985399         0.987941                       0.985399                     0.987941             0.987941
ceiling                           0.995628         0.996236                       0.995628                     0.996236             0.996236

final F0.5 b

In [ ]:
# ===== CELL 9: remaining errors (with the final decision) =====
er.show_errors(CE, p, keep, S, yCE, INPUT_BASE, PARQUET_DIR, n=15)


######################### FALSE MERGES (predicted, not true) #########################
p=0.790  S1-659289780 | Custom Genomic Global Group || 197 Owsley Fork Road, Berea, KY
           S3-455199553 | Custom Genomic Glóbal Ltd || 

p=0.757  S1-486048496 | Tristar Welfare Society Corp || 10-3-316/A, Masab Tank, Asifnagar, Hyderabad, Telangana
           S2-484705208 | Tristar Welfare Society Corp || 

p=0.997  S1-831467241 | Thanjavur Services || No 4, Sekilar St, Muthamil Nagar, Thanjavur, Tamil Nadu
           S2-685810596 | Thanjavur Partners || #4 SEKLIAR STREET MUTHAMIL NAGAR, THANJAVUR, தமிழ்நாடு

p=0.662  S1-857158549 | Hyderabad Management Private Limited || Plot No.2, House No.8-30/P/1 Hema Nagar, Boduppal, Hyderabad, Telangana
           S3-433545160 | Hyderabad  Management Limited Services || P 2, Hyderabad, Ghmc M Corp Og, తెలంగాణ

p=0.955  S1-115201857 | Tri-State Initiative IV || 226 Orange Street, Duluth, MN
           S2-809399826 | Tri-State Initiative IU || MN, 00226 O

In [ ]:
# ===== CELL 10: final models on ALL validation rows -> FINAL_DIR (used by test_v4.ipynb) =====
import lightgbm as lgb
m1p, m2p = f"{FINAL_DIR}/model_stage1.txt", f"{FINAL_DIR}/model_stage2.txt"
if not os.path.exists(m1p):
    r1 = int(np.mean(cv1["iters"]) * 1.1) + 1
    m1 = er.train_final(X1, y1, er.FEATURES_S1, num_rounds=r1); m1.save_model(m1p); del m1; gc.collect()
if not os.path.exists(m2p):
    r2 = int(np.mean(cv2["iters"]) * 1.1) + 1
    m2 = er.train_final(X2, yCE[rows], er.FEATURES_S2, num_rounds=r2); m2.save_model(m2p); del m2; gc.collect()
shutil.copy(f"{RUN_DIR}/metrics_v4.json", f"{FINAL_DIR}/metrics_v4.json")
shutil.copy(f"{WORK_DIR}/prep/train_vocab.json", f"{FINAL_DIR}/train_vocab.json")
shutil.copy(f"{WORK_DIR}/syllables_train.json", f"{FINAL_DIR}/syllables_train.json")
print("final bundle:", sorted(os.listdir(FINAL_DIR)))
print("-> download/attach", FINAL_DIR, "as a Kaggle dataset for test_v4.ipynb")

[20:57:39] [final model] 91 features, trained on 10,194,533 rows (1328s)
[21:00:14] [final model] 107 features, trained on 1,254,438 rows (155s)
final bundle: ['metrics_v4.json', 'model_stage1.txt', 'model_stage2.txt', 'syllables_train.json', 'train_vocab.json']
-> download/attach /content/drive/MyDrive/amazon ml new arch/er_work/final_v4 as a Kaggle dataset for test_v4.ipynb
